# Spirometry analysis

This sector provide simulation of spirometry predictions, including FEV-1 (Forced Expiratory Volume in 1 second) and FVC (Forced vital capacity). Then comparing the ratio of FEV-1/FVC to 0.7 to diagnose whether a patient is having COPD (Chronic Obstructive Pulmonary Disease).

In [1]:
import numpy as np
import pandas as pd
from patsy import dmatrix

# Sample input matrices (each: patients × timepoints)
# Replace with your actual data
ages = np.array([[20, 21, 22], [30, 31, 32]])              # shape (2, 3)
BMIs = np.array([[22, 23, 24], [25, 26, 27]])
heights = np.array([[170, 170, 170], [165, 165, 165]])
smoking_years = np.array([[0, 1, 2], [10, 11, 12]])

# Step 1: Flatten matrices into long-form data
data = pd.DataFrame({
    'Age_in_years': ages.flatten(),
    'BMI': BMIs.flatten(),
    'Height_cm': heights.flatten(),
    'smoking_years': smoking_years.flatten(),
})

# Get min and max for boundary knots
boundary_knots = (data['Age_in_years'].min(), data['Age_in_years'].max())

X_spline = dmatrix(
    "cr(Age_in_years, knots=(23, 24))",
    data=data,
    return_type="dataframe"
)

X_spline = X_spline.iloc[:, [1, 2, 3]]

X = pd.concat([
    dmatrix("BMI + I(BMI**2) + Height_cm + I(Height_cm**2) + smoking_years", data=data, return_type="dataframe"),
    X_spline
], axis=1)

print(X.columns)

Index(['Intercept', 'BMI', 'I(BMI ** 2)', 'Height_cm', 'I(Height_cm ** 2)',
       'smoking_years', 'cr(Age_in_years, knots=(23, 24))[0]',
       'cr(Age_in_years, knots=(23, 24))[1]',
       'cr(Age_in_years, knots=(23, 24))[2]'],
      dtype='object')


In [29]:
coefs = {
    'Intercept': 4406.45425,
    'BMI': 47.21165,
    'I(BMI ** 2)': -0.84642,
    'Height_cm': -69.43769,
    'I(Height_cm ** 2)': 0.32974,
    'smoking_years': -7.60547,
    'cr(Age_in_years, knots=(23, 24))[0]': 270.20024,
    'cr(Age_in_years, knots=(23, 24))[1]': 1848.46398,
    'cr(Age_in_years, knots=(23, 24))[2]': -878.22614
    }
coefs_series = pd.Series(coefs)

In [ ]:
print("X columns:", X.columns)
print("coefs index (excluding Intercept):", coefs_series.drop('Intercept').index)

# Compute prediction
pred = X.drop('Intercept') @ coefs_series.drop('Intercept') + coefs_series['Intercept']

#123456

# Reshape back to (n_patients, n_years)
n_patients, n_years = ages.shape
fev1_preds = pred.values.reshape(n_patients, n_years)

X columns: Index(['Intercept', 'BMI', 'I(BMI ** 2)', 'Height_cm', 'I(Height_cm ** 2)',
       'smoking_years', 'cr(Age_in_years, knots=(23, 24))[0]',
       'cr(Age_in_years, knots=(23, 24))[1]',
       'cr(Age_in_years, knots=(23, 24))[2]'],
      dtype='object')
coefs index (excluding Intercept): Index(['BMI', 'I(BMI ** 2)', 'Height_cm', 'I(Height_cm ** 2)', 'smoking_years',
       'cr(Age_in_years, knots=(23, 24))[0]',
       'cr(Age_in_years, knots=(23, 24))[1]',
       'cr(Age_in_years, knots=(23, 24))[2]'],
      dtype='object')


KeyError: "['Intercept'] not found in axis"